# Arm Grab Tuning

Standalone notebook for tuning the grab sequence. It does not use camera, DepthNet, object detection, or tracked-base movement.

Run the setup/code cells first, then use the compact UI cell at the bottom.

In [ ]:
import json
import time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# Start safe: True means print/log only. Set False when you are ready to move the real arm.
DRY_RUN_ARM = True
PARAM_PATH = Path('arm_grab_tuning_params.json')

ttl_servo = None
if not DRY_RUN_ARM:
    from SCSCtrl import TTLServo
    ttl_servo = TTLServo

DEFAULT_PARAMS = {
    'safe_s1': 0,
    'safe_s2': 0,
    'safe_s3': 0,
    'safe_s4': 0,
    'safe_s5': 0,
    'ready_s1': 0,
    'ready_s2': 18,
    'ready_s3': -12,
    'ready_s4': 0,
    'ready_s5': 0,
    'pick_s1': 0,
    'pick_s2': 5,
    'pick_s3': 18,
    'pick_s4': -55,
    'pick_s5': 0,
    'lift_state_s1': 0,
    'lift_state_s2': 0,
    'lift_state_s3': 28,
    'lift_state_s4': -55,
    'lift_state_s5': 0,
    'pre_s2': 18,
    'pre_s3': -12,
    'reach_s2': 26,
    'reach_s3': -22,
    'lift_s2': 5,
    'lift_s3': 18,
    'gripper_open': 0,
    'gripper_close_1': -45,
    'gripper_close_2': -55,
    'arm_speed': 80,
    'gripper_speed': 120,
    'settle_seconds': 0.35,
}

def load_params():
    if PARAM_PATH.exists():
        data = json.loads(PARAM_PATH.read_text())
        params = dict(DEFAULT_PARAMS)
        params.update(data)
        return params
    return dict(DEFAULT_PARAMS)

params = load_params()
print('DRY_RUN_ARM =', DRY_RUN_ARM)
print('params loaded from', PARAM_PATH if PARAM_PATH.exists() else 'defaults')


In [ ]:
def angle_slider(name, min_value=-180, max_value=180):
    return widgets.IntSlider(
        value=int(params[name]), min=min_value, max=max_value, step=1,
        description=name, continuous_update=False, style={'description_width': '120px'},
        layout=widgets.Layout(width='520px')
    )

def speed_slider(name):
    return widgets.IntSlider(
        value=int(params[name]), min=20, max=300, step=5,
        description=name, continuous_update=False, style={'description_width': '120px'},
        layout=widgets.Layout(width='520px')
    )

def float_slider(name):
    return widgets.FloatSlider(
        value=float(params[name]), min=0.05, max=1.5, step=0.05,
        description=name, continuous_update=False, style={'description_width': '120px'},
        layout=widgets.Layout(width='520px')
    )

sliders = {
    'safe_s1': angle_slider('safe_s1'),
    'safe_s2': angle_slider('safe_s2'),
    'safe_s3': angle_slider('safe_s3'),
    'safe_s4': angle_slider('safe_s4'),
    'safe_s5': angle_slider('safe_s5'),
    'ready_s1': angle_slider('ready_s1'),
    'ready_s2': angle_slider('ready_s2'),
    'ready_s3': angle_slider('ready_s3'),
    'ready_s4': angle_slider('ready_s4'),
    'ready_s5': angle_slider('ready_s5'),
    'pick_s1': angle_slider('pick_s1'),
    'pick_s2': angle_slider('pick_s2'),
    'pick_s3': angle_slider('pick_s3'),
    'pick_s4': angle_slider('pick_s4'),
    'pick_s5': angle_slider('pick_s5'),
    'lift_state_s1': angle_slider('lift_state_s1'),
    'lift_state_s2': angle_slider('lift_state_s2'),
    'lift_state_s3': angle_slider('lift_state_s3'),
    'lift_state_s4': angle_slider('lift_state_s4'),
    'lift_state_s5': angle_slider('lift_state_s5'),
    'pre_s2': angle_slider('pre_s2'),
    'pre_s3': angle_slider('pre_s3'),
    'reach_s2': angle_slider('reach_s2'),
    'reach_s3': angle_slider('reach_s3'),
    'lift_s2': angle_slider('lift_s2'),
    'lift_s3': angle_slider('lift_s3'),
    'gripper_open': angle_slider('gripper_open'),
    'gripper_close_1': angle_slider('gripper_close_1'),
    'gripper_close_2': angle_slider('gripper_close_2'),
    'arm_speed': speed_slider('arm_speed'),
    'gripper_speed': speed_slider('gripper_speed'),
    'settle_seconds': float_slider('settle_seconds'),
}

def current_params():
    return {name: slider.value for name, slider in sliders.items()}

def save_params(_=None):
    data = current_params()
    PARAM_PATH.write_text(json.dumps(data, indent=2))
    print('[params] saved to', PARAM_PATH)

def move_servo(servo_id, angle, speed, label=''):
    print('[arm] servo={} angle={} speed={} {}'.format(servo_id, angle, speed, label))
    if ttl_servo is not None:
        ttl_servo.servoAngleCtrl(int(servo_id), int(angle), 1, int(speed))
    time.sleep(float(sliders['settle_seconds'].value))

def apply_pose(name, pose, speed):
    print('[arm] pose:', name)
    for servo_id, angle in pose:
        move_servo(servo_id, angle, speed, name)

def safe_home():
    p = current_params()
    pose = [(1, p['safe_s1']), (2, p['safe_s2']), (3, p['safe_s3']), (4, p['safe_s4']), (5, p['safe_s5'])]
    apply_pose('safe_home', pose, p['arm_speed'])

def ready_state():
    p = current_params()
    pose = [(1, p['ready_s1']), (2, p['ready_s2']), (3, p['ready_s3']), (4, p['ready_s4']), (5, p['ready_s5'])]
    apply_pose('ready_state', pose, p['arm_speed'])

def pick_state():
    p = current_params()
    pose = [(1, p['pick_s1']), (2, p['pick_s2']), (3, p['pick_s3']), (4, p['pick_s4']), (5, p['pick_s5'])]
    apply_pose('pick_state', pose, p['arm_speed'])


def lift_state():
    p = current_params()
    pose = [(1, p['lift_state_s1']), (2, p['lift_state_s2']), (3, p['lift_state_s3']), (4, p['lift_state_s4']), (5, p['lift_state_s5'])]
    apply_pose('lift_state', pose, p['arm_speed'])


def open_gripper():
    p = current_params()
    move_servo(4, p['gripper_open'], p['gripper_speed'], 'open_gripper')

def close_gripper():
    p = current_params()
    move_servo(4, p['gripper_close_1'], p['gripper_speed'], 'close_1')
    time.sleep(0.5)
    move_servo(4, p['gripper_close_2'], p['gripper_speed'], 'close_2')

def run_grab_sequence(_=None):
    p = current_params()
    print('[flow] grab sequence start')
    try:
        safe_home()
        ready_state()
        open_gripper()
        apply_pose('pre_grasp', [(2, p['pre_s2']), (3, p['pre_s3'])], p['arm_speed'])
        apply_pose('reach', [(2, p['reach_s2']), (3, p['reach_s3'])], p['arm_speed'])
        close_gripper()
        pick_state()
        lift_state()
        print('[flow] grab sequence done')
    finally:
        print('[flow] final safe_home')
        safe_home()

save_button = widgets.Button(description='Save Params', button_style='info')
home_button = widgets.Button(description='Safe Home', button_style='warning')
ready_button = widgets.Button(description='Ready State', button_style='success')
pick_button = widgets.Button(description='Pick State')
lift_state_button = widgets.Button(description='Lift State')
open_button = widgets.Button(description='Open Gripper')
close_button = widgets.Button(description='Close Gripper')
run_button = widgets.Button(description='Run Grab Sequence', button_style='danger')

save_button.on_click(save_params)
home_button.on_click(lambda _: safe_home())
ready_button.on_click(lambda _: ready_state())
pick_button.on_click(lambda _: pick_state())
lift_state_button.on_click(lambda _: lift_state())
open_button.on_click(lambda _: open_gripper())
close_button.on_click(lambda _: close_gripper())
run_button.on_click(run_grab_sequence)

print('UI objects ready')


In [ ]:
button_row = widgets.HBox([save_button, home_button, ready_button, pick_button, lift_state_button, open_button, close_button, run_button])

ui = widgets.VBox([
    widgets.HTML('<b>Actions</b>'),
    button_row,
    widgets.HTML('<b>Safe Home</b>'),
    sliders['safe_s1'], sliders['safe_s2'], sliders['safe_s3'], sliders['safe_s4'], sliders['safe_s5'],
    widgets.HTML('<b>Ready State</b>'),
    sliders['ready_s1'], sliders['ready_s2'], sliders['ready_s3'], sliders['ready_s4'], sliders['ready_s5'],
    widgets.HTML('<b>Pick State</b>'),
    sliders['pick_s1'], sliders['pick_s2'], sliders['pick_s3'], sliders['pick_s4'], sliders['pick_s5'],
    widgets.HTML('<b>Lift State</b>'),
    sliders['lift_state_s1'], sliders['lift_state_s2'], sliders['lift_state_s3'], sliders['lift_state_s4'], sliders['lift_state_s5'],
    widgets.HTML('<b>Pre Grasp / Reach</b>'),
    sliders['pre_s2'], sliders['pre_s3'], sliders['reach_s2'], sliders['reach_s3'],
    widgets.HTML('<b>Gripper</b>'),
    sliders['gripper_open'], sliders['gripper_close_1'], sliders['gripper_close_2'],
    widgets.HTML('<b>Speed / Timing</b>'),
    sliders['arm_speed'], sliders['gripper_speed'], sliders['settle_seconds'],
])

display(ui)


Tip: first run with `DRY_RUN_ARM = True`. When the printed sequence looks right, restart the kernel, set `DRY_RUN_ARM = False`, and test small moves. If any pose is unsafe, stop and adjust the sliders before running the full sequence again.